# 06 - Deep Agent com memória (AGENTS.md)

Um arquivo `AGENTS.md` guarda as regras de um projeto: idioma, formato de resposta, convenções. O agente lê esse arquivo antes de responder, e as regras passam a valer sem precisar repetir no prompt de cada tarefa.

Diferença para as skills do notebook anterior:

- **Skill**: instrução para uma tarefa específica. O agente lê quando a tarefa combina com a descrição.
- **AGENTS.md**: regras gerais do projeto. O agente carrega o arquivo sempre, antes de qualquer tarefa.

Este notebook compara duas execuções da mesma tarefa:

1. **sem memória**: o agente não recebe o `AGENTS.md`;
2. **com memória**: o agente recebe `memoria/AGENTS.md` com `memory=[...]`.

A diferença aparece no formato da resposta. Os dados e as ferramentas são os mesmos.

Tudo fica registrado no MLflow, no experimento `deep-agent-memoria`.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook.

In [1]:
import ast
import json
import operator
import os
import unicodedata
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-memoria"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. O arquivo AGENTS.md

O arquivo fica em `memoria/AGENTS.md`. Abaixo, o conteúdo que o agente vai receber. Ele é lido do disco e carregado no estado do agente, como as skills no notebook 05.

In [2]:
TEXTO_AGENTS = Path("memoria/AGENTS.md").read_text(encoding="utf-8")
display(Markdown(TEXTO_AGENTS))

ARQUIVOS_MEMORIA = {"/memoria/AGENTS.md": create_file_data(TEXTO_AGENTS)}

# Instruções do projeto

Estas regras valem para todas as respostas do agente de vendas deste projeto.

## Idioma e estilo

- Responda sempre em português do Brasil.
- Não use emojis.

## Formato das respostas

- Mostre totais por produto em uma tabela Markdown, com as colunas `Produto | Unidades`.
- Escreva números grandes com separador de milhar com ponto, por exemplo `1.428`.
- Termine a resposta com a linha `Fonte: dados de consultar_vendas`.


## 3. Ferramentas

- `consultar_vendas`: devolve as vendas de um mês. Os dados são simulados.
- `calcular`: faz contas com números e as quatro operações, usando `ast`. Nunca use `eval` com texto gerado por LLM.

In [3]:
VENDAS_API = {
    "janeiro": {"cafe": 120, "pao": 300, "queijo": 45},
    "fevereiro": {"cafe": 135, "pao": 280, "queijo": 50},
    "marco": {"cafe": 150, "pao": 310, "queijo": 38},
}


@tool
def consultar_vendas(mes: str) -> str:
    """Consulta as vendas de um mês do primeiro trimestre. Retorna a quantidade vendida de cada produto em JSON. Meses válidos: janeiro, fevereiro, março."""
    chave = unicodedata.normalize("NFKD", mes.lower().strip()).encode("ascii", "ignore").decode()
    dados = VENDAS_API.get(chave)
    if dados is None:
        return json.dumps({"mes": mes, "erro": "mês não encontrado"}, ensure_ascii=False)
    return json.dumps({"mes": mes, "vendas": dados}, ensure_ascii=False)


OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '(120 + 135) * 2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


FERRAMENTAS = [consultar_vendas, calcular]

print(consultar_vendas.name, "|", calcular.name)

consultar_vendas | calcular


## 4. Criando os dois agentes

Os dois agentes são iguais, exceto por uma linha: o segundo recebe `memory=["/memoria/AGENTS.md"]`. O `system_prompt` é o mesmo e não menciona formato, idioma nem fonte. Assim, qualquer diferença na resposta vem do `AGENTS.md`.

In [4]:
PROMPT_SISTEMA = (
    "Você é um analista de vendas. Em tarefas com etapas, a primeira ação é chamar write_todos "
    "com um plano de 3 a 5 passos. Use as ferramentas para buscar os dados e fazer as contas."
)

agente_sem_memoria = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    system_prompt=PROMPT_SISTEMA,
)

agente_com_memoria = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    memory=["/memoria/AGENTS.md"],
    system_prompt=PROMPT_SISTEMA,
)

## 5. Executando a mesma tarefa nos dois agentes

A tarefa é a mesma para os dois. A função `resposta_final` devolve só o texto da última mensagem do agente, para comparar.

In [5]:
TAREFA = "Quantas unidades de cada produto foram vendidas no primeiro trimestre? Diga também qual foi o mais vendido."


def resposta_final(estado: dict) -> str:
    for mensagem in reversed(estado["messages"]):
        if isinstance(mensagem, AIMessage) and mensagem.content and not mensagem.tool_calls:
            return mensagem.content
    return ""


estado_sem = agente_sem_memoria.invoke({"messages": [HumanMessage(content=TAREFA)]})
trace_sem = mlflow.get_last_active_trace_id()

estado_com = agente_com_memoria.invoke({
    "messages": [HumanMessage(content=TAREFA)],
    "files": ARQUIVOS_MEMORIA,
})
trace_com = mlflow.get_last_active_trace_id()

display(Markdown(f"### Sem memória\n\n{resposta_final(estado_sem)}"))
display(Markdown(f"### Com memória\n\n{resposta_final(estado_com)}"))

### Sem memória

Vendas no 1º trimestre (janeiro + fevereiro + março):

- Café: 405 unidades
- Pão: 890 unidades
- Queijo: 133 unidades

Produto mais vendido: Pão (890 unidades).

### Com memória

Produto | Unidades
--- | ---
cafe | 405
pao | 890
queijo | 133

Mais vendido: pao (890 unidades)

Fonte: dados de consultar_vendas

## 6. Comparando as regras

Cada regra do `AGENTS.md` vira uma checagem simples no texto da resposta. A comparação mostra o efeito do arquivo, sem depender de leitura manual.

In [6]:
def checar(texto: str) -> dict:
    return {
        "tabela Markdown": "Produto | Unidades" in texto and "---" in texto and "|" in texto,
        "linha de fonte": "Fonte: dados de consultar_vendas" in texto,
        "sem emojis": not any(ord(c) > 0x2600 for c in texto),
    }


resultados = {
    "sem memória": checar(resposta_final(estado_sem)),
    "com memória": checar(resposta_final(estado_com)),
}

for regra in resultados["sem memória"]:
    sem = "sim" if resultados["sem memória"][regra] else "não"
    com = "sim" if resultados["com memória"][regra] else "não"
    print(f"{regra:<18} sem memória: {sem:<4} com memória: {com}")

tabela Markdown    sem memória: não  com memória: sim
linha de fonte     sem memória: não  com memória: sim
sem emojis         sem memória: sim  com memória: sim


## 7. Acompanhando o rastreamento no MLflow

As duas execuções viraram traces no experimento `deep-agent-memoria`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-memoria` e abra a aba **Traces**. Há dois traces: um sem memória e outro com memória.
5. Compare os dois. No trace com memória, o modelo recebe o conteúdo do `AGENTS.md` no prompt. Dá para ver isso nos spans `CHAT_MODEL`, nas mensagens de entrada.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista os traces e os spans pelo notebook.

In [7]:
# Mostra só os traces desta execução, pelos ids guardados acima.
for nome, trace_id in [("sem memória", trace_sem), ("com memória", trace_com)]:
    trace = mlflow.get_trace(trace_id)
    spans = trace.data.spans
    modelos = [span for span in spans if span.span_type == "CHAT_MODEL"]
    ferramentas = [span.name for span in spans if span.span_type == "TOOL"]
    print(f"Trace {nome} ({trace_id}): {len(modelos)} chamadas ao modelo")
    print(f"  ferramentas: {', '.join(ferramentas)}\n")

Trace sem memória (tr-1ff142178ef5623dcd5dd6fae1f29915): 11 chamadas ao modelo
  ferramentas: write_todos, consultar_vendas, consultar_vendas, consultar_vendas, write_todos, calcular, calcular, calcular, write_todos, write_todos

Trace com memória (tr-14be9014502f95adf9bc56954c348070): 12 chamadas ao modelo
  ferramentas: write_todos, consultar_vendas, write_todos, consultar_vendas, write_todos, consultar_vendas, write_todos, calcular, calcular, calcular, write_todos



## Observações

- `AGENTS.md` é um arquivo de regras do projeto, lido sempre. Não é uma instrução para uma tarefa, como as skills.
- Regras que mudam o formato, como tabela ou fonte, funcionam melhor escritas como exigências específicas, do jeito que estão no arquivo.
- Para trocar as regras, edite `memoria/AGENTS.md`. O agente não precisa mudar.
- O resultado pode variar entre execuções. Se uma checagem falhar, compare a resposta com o `AGENTS.md` antes de concluir que a memória não funcionou.